# Assignment 4 – Portugal | Part 2: PyPSA Model + Sensitivity Analysis

**Requires outputs from `EAnalysis.ipynb`** (run that notebook first).

In [2]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, geopandas as gpd
import pypsa, xarray as xr, requests
from pypsa.costs import annuity

url_dataset = 'https://tubcloud.tu-berlin.de/s/567ckizz2Y6RLQq'
url_tech_cost = 'https://raw.githubusercontent.com/PyPSA/technology-data/master/outputs/costs_2030.csv'
url_load = f'{url_dataset}/download?path=%2Fgegis&files=load.csv'
url_pp   = f'{url_dataset}/download?path=%2Fglobal-power-plant-database&files=global_power_plant_database.csv'

pypsa.options.params.optimize.include_objective_constant = True
print('Imports done.')

Imports done.


## Load EAnalysis outputs

In [3]:
regions          = gpd.read_file('regions.gpkg')
regions          = regions.set_index('nuts2')
regions['representative_point'] = regions.geometry.representative_point()

offshore_valid   = gpd.read_file('offshore_valid.gpkg').set_index('nuts2')

cf_wind          = xr.open_dataarray('cf_wind.nc')
cf_solar         = xr.open_dataarray('cf_solar.nc')
cf_offshore      = xr.open_dataarray('cf_offshore.nc')

p_nom_max_wind     = pd.read_csv('p_nom_max_wind.csv',     index_col=0)['p_nom_max_MW']
p_nom_max_solar    = pd.read_csv('p_nom_max_solar.csv',    index_col=0)['p_nom_max_MW']
p_nom_max_offshore = pd.read_csv('p_nom_max_offshore.csv', index_col=0)['p_nom_max_MW']

print('Regions:', list(regions.index))
print('CF wind dim:', cf_wind.dims)
print('p_nom_max_offshore:\n', p_nom_max_offshore)

Regions: ['Alentejo', 'Algarve', 'Centro', 'Lisboa', 'Norte']
CF wind dim: ('time', 'nuts2')
p_nom_max_offshore:
 nuts2
Alentejo     5512.750661
Algarve     83465.168955
Centro      65569.599010
Lisboa      58028.784018
Norte       39911.710456
Name: p_nom_max_MW, dtype: float64


## 1. Technology costs (2030)

In [4]:
costs = pd.read_csv(url_tech_cost, index_col=[0, 1])

# kW -> MW
costs.loc[costs.unit.str.contains('/kW'), 'value'] *= 1e3
costs.unit = costs.unit.str.replace('/kW', '/MW')

defaults = {'FOM': 0, 'VOM': 0, 'efficiency': 1, 'fuel': 0, 'investment': 0,
            'lifetime': 25, 'CO2 intensity': 0, 'discount rate': 0.07}
costs = costs.value.unstack().fillna(defaults)

# Gas carriers inherit fuel cost + CO2 from 'gas'
for t in ['OCGT', 'CCGT']:
    costs.at[t, 'fuel']          = costs.at['gas', 'fuel']
    costs.at[t, 'CO2 intensity'] = costs.at['gas', 'CO2 intensity']

costs['marginal_cost'] = costs['VOM'] + costs['fuel'] / costs['efficiency']
annuity_factor = annuity(costs['discount rate'], costs['lifetime'])
costs['capital_cost'] = (annuity_factor + costs['FOM'] / 100) * costs['investment']

check = ['solar', 'onwind', 'offwind', 'offwind-float', 'OCGT', 'CCGT',
         'battery storage', 'battery inverter', 'electrolysis', 'fuel cell',
         'hydrogen storage underground']
costs.loc[check, ['investment', 'lifetime', 'marginal_cost', 'capital_cost']].round(1)

parameter,investment,lifetime,marginal_cost,capital_cost
technology,,,,
solar,683146.2,40.0,0.0,64560.1
onwind,1383305.9,30.0,1.8,128306.3
offwind,2114991.0,30.0,0.0,219475.6
offwind-float,2954736.3,20.0,0.0,312885.7
OCGT,581394.9,25.0,75.3,60235.7
CCGT,1108716.6,25.0,54.6,132274.9
battery storage,189861.0,25.0,0.0,16292.1
battery inverter,213927.9,10.0,0.0,31180.5
electrolysis,1886001.9,25.0,0.0,237278.9


## 2. Load time series

In [5]:
load_raw    = pd.read_csv(url_load, index_col=0, parse_dates=True)
load_pt     = load_raw['PT'].loc['2013'].resample('3h').mean()
population  = pd.Series({'Norte': 3689, 'Centro': 2227, 'Lisboa': 2821,
                          'Alentejo': 757, 'Algarve': 451})
pop_share   = population / population.sum()
load_region = pd.DataFrame({r: load_pt * pop_share[r] for r in regions.index})
print('Load shape:', load_region.shape)
load_region.head(2)

Load shape: (2920, 5)


,Alentejo,Algarve,Centro,Lisboa,Norte
time,,,,,
2013-01-01 00:00:00,443.115591,263.996211,1303.591044,1651.293370,2159.383638
2013-01-01 03:00:00,416.359499,248.055659,1224.877944,1551.585397,2028.996289


## 3. Existing power plants

In [6]:
pp_raw  = pd.read_csv(url_pp, low_memory=False)
pp_pt   = pp_raw[(pp_raw['country'] == 'PRT') &
                  ~pp_raw['primary_fuel'].isin(['Wind', 'Solar'])].copy()
pp_pt   = pp_pt.dropna(subset=['latitude', 'longitude'])
pp_gdf  = gpd.GeoDataFrame(pp_pt,
           geometry=gpd.points_from_xy(pp_pt.longitude, pp_pt.latitude),
           crs=4326).to_crs(3035)
pp_join = gpd.sjoin_nearest(pp_gdf, regions[['geometry']].reset_index(),
                             how='left', distance_col='dist')

hydro     = pp_join[pp_join['primary_fuel'] == 'Hydro']
hydro_cap = hydro.groupby('nuts2')['capacity_mw'].sum()

# Use estimated generation if actual 2013 column is missing/all-NaN
gen_col = 'generation_gwh_2013'
if gen_col not in hydro.columns or hydro[gen_col].isna().all():
    gen_col = 'estimated_generation_gwh_2013'
hydro_gen = hydro.groupby('nuts2')[gen_col].sum()
hydro_cf  = (hydro_gen * 1000 / (hydro_cap * 8760)).clip(0, 1).fillna(0.15)

non_hydro    = pp_join[pp_join['primary_fuel'] != 'Hydro']
conventional = non_hydro.groupby(['nuts2', 'primary_fuel'])['capacity_mw'] \
                         .sum().unstack(fill_value=0)

print('Hydro CF:\n', hydro_cf.round(3))
print('Conventional [MW]:\n', conventional)

Hydro CF:
 nuts2
Alentejo    0.366
Algarve     0.332
Centro      0.322
Lisboa      0.342
Norte       0.311
dtype: float64
Conventional [MW]:
 primary_fuel  Biomass    Coal     Gas  Geothermal  Waste
nuts2                                                   
Algarve           0.0     0.0     0.0         0.0   10.9
Centro          347.7   682.0  1663.0        28.8   18.5
Lisboa           66.4  1296.0  1176.0         0.0   62.2
Norte            38.8     0.0   990.0         0.0   40.2


## 4. `build_network()` function

In [7]:
def build_network(solar_cost_factor=1.0):
    n = pypsa.Network()
    n.set_snapshots(load_region.index)

    # Buses
    for region, row in regions.iterrows():
        pt = row['representative_point']
        n.add('Bus', region, x=pt.x, y=pt.y, carrier='AC')

    # Loads
    for region in regions.index:
        n.add('Load', f'load-{region}', bus=region, p_set=load_region[region])

    # Hydro (fixed, constant CF)
    for region in regions.index:
        cap = float(hydro_cap.get(region, 0))
        if cap > 0:
            n.add('Generator', f'hydro-{region}', bus=region, carrier='hydro',
                  p_nom=cap, p_nom_extendable=False,
                  p_max_pu=float(hydro_cf.get(region, 0.15)),
                  marginal_cost=0., capital_cost=0.)

    # CO2 carriers
    n.add('Carrier', 'gas',     co2_emissions=costs.at['gas',  'CO2 intensity'])
    n.add('Carrier', 'coal',    co2_emissions=costs.at['coal', 'CO2 intensity'])
    n.add('Carrier', 'biomass', co2_emissions=0.)
    n.add('Carrier', 'waste',   co2_emissions=0.)

    # Existing conventional fleet (Gas/Coal have CO2; Biomass/Waste are zero-emission)
    biomass_mc = costs.at['biomass', 'marginal_cost'] if 'biomass' in costs.index else 3.0
    fuel_map = {
        'Gas':     ('gas',     costs.at['CCGT', 'marginal_cost']),
        'Coal':    ('coal',    costs.at['coal', 'marginal_cost']),
        'Biomass': ('biomass', biomass_mc),
        'Waste':   ('waste',   biomass_mc),
    }
    for fuel, (carrier, mc) in fuel_map.items():
        if fuel not in conventional.columns: continue
        for region in regions.index:
            if region not in conventional.index: continue
            cap = float(conventional.at[region, fuel])
            if cap > 0:
                n.add('Generator', f'{fuel.lower()}-{region}', bus=region,
                      carrier=carrier, p_nom=cap, p_nom_extendable=False,
                      marginal_cost=mc, capital_cost=0.)

    # Renewable generators (extendable)
    solar_cc = costs.at['solar', 'capital_cost'] * solar_cost_factor
    for region in regions.index:
        n.add('Generator', f'onwind-{region}', bus=region, carrier='onwind',
              p_nom_extendable=True,
              p_nom_max=float(p_nom_max_wind.get(region, 0)),
              p_max_pu=cf_wind.sel(nuts2=region).values,
              capital_cost=costs.at['onwind', 'capital_cost'],
              marginal_cost=costs.at['onwind', 'marginal_cost'])
        n.add('Generator', f'solar-{region}', bus=region, carrier='solar',
              p_nom_extendable=True,
              p_nom_max=float(p_nom_max_solar.get(region, 0)),
              p_max_pu=cf_solar.sel(nuts2=region).values,
              capital_cost=solar_cc,
              marginal_cost=costs.at['solar', 'marginal_cost'])
        if float(p_nom_max_offshore.get(region, 0)) > 0:
            n.add('Generator', f'offwind-{region}', bus=region, carrier='offwind',
                  p_nom_extendable=True,
                  p_nom_max=float(p_nom_max_offshore.get(region, 0)),
                  p_max_pu=cf_offshore.sel(nuts2=region).values,
                  capital_cost=costs.at['offwind-float', 'capital_cost'],
                  marginal_cost=costs.at['offwind-float', 'marginal_cost'])
        n.add('Generator', f'OCGT-{region}', bus=region, carrier='gas',
              p_nom_extendable=True,
              capital_cost=costs.at['OCGT', 'capital_cost'],
              marginal_cost=costs.at['OCGT', 'marginal_cost'])

    # Transmission links
    for r1, r2 in [('Norte','Centro'), ('Centro','Lisboa'), ('Centro','Alentejo'),
                   ('Lisboa','Alentejo'), ('Alentejo','Algarve')]:
        pt1 = regions.at[r1, 'representative_point']
        pt2 = regions.at[r2, 'representative_point']
        dist_km = pt1.distance(pt2) / 1000 * 1.5
        n.add('Link', f'line-{r1}-{r2}', bus0=r1, bus1=r2, p_min_pu=-1,
              p_nom_extendable=True, capital_cost=700 * dist_km, marginal_cost=0.)

    # Battery storage (4h)
    batt_cc = costs.at['battery inverter', 'capital_cost']
    batt_e_cc = costs.at['battery storage', 'capital_cost']
    for region in regions.index:
        n.add('StorageUnit', f'battery-{region}', bus=region, carrier='battery',
              p_nom_extendable=True, max_hours=4,
              capital_cost=batt_cc + 4 * batt_e_cc,
              efficiency_store=costs.at['battery inverter', 'efficiency'],
              efficiency_dispatch=costs.at['battery inverter', 'efficiency'],
              cyclic_state_of_charge=True)

    # Hydrogen storage (168h)
    h2_power_cc = costs.at['electrolysis', 'capital_cost'] + costs.at['fuel cell', 'capital_cost']
    h2_energy_cc = costs.at['hydrogen storage underground', 'capital_cost']
    for region in regions.index:
        n.add('StorageUnit', f'H2-{region}', bus=region, carrier='H2',
              p_nom_extendable=True, max_hours=168,
              capital_cost=h2_power_cc + 168 * h2_energy_cc,
              efficiency_store=costs.at['electrolysis', 'efficiency'],
              efficiency_dispatch=costs.at['fuel cell', 'efficiency'],
              cyclic_state_of_charge=True)
    return n

print('build_network() defined.')
print(f'Solar base capital cost: {costs.at["solar", "capital_cost"]:.0f} EUR/MW/a')

build_network() defined.
Solar base capital cost: 64560 EUR/MW/a


## 5. Run 1 — No CO₂ limit

In [8]:
n_free = build_network(solar_cost_factor=1.0)
n_free.optimize(solver_name='highs')
print('=== No CO2 limit ===')
print(f'Total cost: {n_free.objective/1e9:.2f} bn EUR/yr')
print(n_free.generators.groupby('carrier')['p_nom_opt'].sum().round(0))
n_free.export_to_netcdf('network_free.nc')
print('Saved: network_free.nc')

Index(['Alentejo', 'Algarve', 'Centro', 'Lisboa', 'Norte'], dtype='object', name='name')
Index(['hydro-Alentejo', 'hydro-Algarve', 'hydro-Centro', 'hydro-Lisboa',
       'hydro-Norte', 'onwind-Alentejo', 'solar-Alentejo', 'offwind-Alentejo',
       'onwind-Algarve', 'solar-Algarve', 'offwind-Algarve', 'onwind-Centro',
       'solar-Centro', 'offwind-Centro', 'onwind-Lisboa', 'solar-Lisboa',
       'offwind-Lisboa', 'onwind-Norte', 'solar-Norte', 'offwind-Norte'],
      dtype='object', name='name')
Index(['line-Norte-Centro', 'line-Centro-Lisboa', 'line-Centro-Alentejo',
       'line-Lisboa-Alentejo', 'line-Alentejo-Algarve'],
      dtype='object', name='name')
Index(['battery-Alentejo', 'battery-Algarve', 'battery-Centro',
       'battery-Lisboa', 'battery-Norte', 'H2-Alentejo', 'H2-Algarve',
       'H2-Centro', 'H2-Lisboa', 'H2-Norte'],
      dtype='object', name='name')
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io:Writing objective.
Writing continuous variables.

=== No CO2 limit ===
Total cost: 0.87 bn EUR/yr
carrier
biomass     453.0
coal       1978.0
gas        6017.0
hydro      2760.0
offwind       0.0
onwind        0.0
solar         0.0
waste       132.0
Name: p_nom_opt, dtype: float64
Saved: network_free.nc


## 6. Run 2 — Zero CO₂ (net-zero)

In [9]:
n_zero = build_network(solar_cost_factor=1.0)
n_zero.add('GlobalConstraint', 'co2_limit', sense='<=', constant=0.,
           carrier_attribute='co2_emissions')
n_zero.optimize(solver_name='highs')
print('=== Zero CO2 ===')
print(f'Total cost: {n_zero.objective/1e9:.2f} bn EUR/yr')
print(n_zero.generators.groupby('carrier')['p_nom_opt'].sum().round(0))
n_zero.export_to_netcdf('network_zero.nc')
print('Saved: network_zero.nc')

Index(['Alentejo', 'Algarve', 'Centro', 'Lisboa', 'Norte'], dtype='object', name='name')
Index(['hydro-Alentejo', 'hydro-Algarve', 'hydro-Centro', 'hydro-Lisboa',
       'hydro-Norte', 'onwind-Alentejo', 'solar-Alentejo', 'offwind-Alentejo',
       'onwind-Algarve', 'solar-Algarve', 'offwind-Algarve', 'onwind-Centro',
       'solar-Centro', 'offwind-Centro', 'onwind-Lisboa', 'solar-Lisboa',
       'offwind-Lisboa', 'onwind-Norte', 'solar-Norte', 'offwind-Norte'],
      dtype='object', name='name')
Index(['line-Norte-Centro', 'line-Centro-Lisboa', 'line-Centro-Alentejo',
       'line-Lisboa-Alentejo', 'line-Alentejo-Algarve'],
      dtype='object', name='name')
Index(['battery-Alentejo', 'battery-Algarve', 'battery-Centro',
       'battery-Lisboa', 'battery-Norte', 'H2-Alentejo', 'H2-Algarve',
       'H2-Centro', 'H2-Lisboa', 'H2-Norte'],
      dtype='object', name='name')
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io:Writing objective.
Writing continuous variables.

=== Zero CO2 ===
Total cost: 4.66 bn EUR/yr
carrier
biomass      453.0
coal        1978.0
gas         3829.0
hydro       2760.0
offwind     3130.0
onwind      1998.0
solar      26988.0
waste        132.0
Name: p_nom_opt, dtype: float64


INFO:pypsa.network.io:Exported network 'Unnamed Network' saved to 'network_zero.nc contains: links, sub_networks, storage_units, global_constraints, loads, carriers, buses, generators


Saved: network_zero.nc


## 7. Sensitivity Analysis — Solar PV capital cost

Reduce solar cost from 100% → 75% → 50% → 25% → 0% under net-zero CO₂ constraint.

In [ ]:
solar_factors = [1.0, 0.75, 0.50, 0.25, 0.0]
results = []

for factor in solar_factors:
    print(f'Solving: solar cost = {factor*100:.0f}% ...', flush=True)
    net = build_network(solar_cost_factor=factor)
    net.add('GlobalConstraint', 'co2_limit', sense='<=', constant=0.,
            carrier_attribute='co2_emissions')
    net.optimize(solver_name='highs')

    caps = net.generators.groupby('carrier')['p_nom_opt'].sum()
    stor = net.storage_units.groupby('carrier')['p_nom_opt'].sum()
    results.append({
        'solar_cost_pct':    f'{factor*100:.0f}%',
        'total_cost_bn_eur': net.objective / 1e9,
        'solar_GW':          caps.get('solar',   0) / 1e3,
        'onwind_GW':         caps.get('onwind',  0) / 1e3,
        'offwind_GW':        caps.get('offwind', 0) / 1e3,
        'OCGT_GW':           caps.get('gas',     0) / 1e3,
        'battery_GW':        stor.get('battery', 0) / 1e3,
        'H2_GW':             stor.get('H2',      0) / 1e3,
        'transmission_GW':   net.links['p_nom_opt'].sum() / 1e3,
    })
    print(f'  Cost: {net.objective/1e9:.2f} bn EUR/yr | Solar: {caps.get("solar",0)/1e3:.1f} GW')

df_sens = pd.DataFrame(results).set_index('solar_cost_pct')
df_sens.round(2)

Solving: solar cost = 100% ...


Index(['Alentejo', 'Algarve', 'Centro', 'Lisboa', 'Norte'], dtype='object', name='name')
Index(['hydro-Alentejo', 'hydro-Algarve', 'hydro-Centro', 'hydro-Lisboa',
       'hydro-Norte', 'onwind-Alentejo', 'solar-Alentejo', 'offwind-Alentejo',
       'onwind-Algarve', 'solar-Algarve', 'offwind-Algarve', 'onwind-Centro',
       'solar-Centro', 'offwind-Centro', 'onwind-Lisboa', 'solar-Lisboa',
       'offwind-Lisboa', 'onwind-Norte', 'solar-Norte', 'offwind-Norte'],
      dtype='object', name='name')
Index(['line-Norte-Centro', 'line-Centro-Lisboa', 'line-Centro-Alentejo',
       'line-Lisboa-Alentejo', 'line-Alentejo-Algarve'],
      dtype='object', name='name')
Index(['battery-Alentejo', 'battery-Algarve', 'battery-Centro',
       'battery-Lisboa', 'battery-Norte', 'H2-Alentejo', 'H2-Algarve',
       'H2-Centro', 'H2-Lisboa', 'H2-Norte'],
      dtype='object', name='name')
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io:Writing objective.
Writing continuous variables.

  Cost: 4.66 bn EUR/yr | Solar: 27.0 GW
Solving: solar cost = 75% ...


Index(['Alentejo', 'Algarve', 'Centro', 'Lisboa', 'Norte'], dtype='object', name='name')
Index(['hydro-Alentejo', 'hydro-Algarve', 'hydro-Centro', 'hydro-Lisboa',
       'hydro-Norte', 'onwind-Alentejo', 'solar-Alentejo', 'offwind-Alentejo',
       'onwind-Algarve', 'solar-Algarve', 'offwind-Algarve', 'onwind-Centro',
       'solar-Centro', 'offwind-Centro', 'onwind-Lisboa', 'solar-Lisboa',
       'offwind-Lisboa', 'onwind-Norte', 'solar-Norte', 'offwind-Norte'],
      dtype='object', name='name')
Index(['line-Norte-Centro', 'line-Centro-Lisboa', 'line-Centro-Alentejo',
       'line-Lisboa-Alentejo', 'line-Alentejo-Algarve'],
      dtype='object', name='name')
Index(['battery-Alentejo', 'battery-Algarve', 'battery-Centro',
       'battery-Lisboa', 'battery-Norte', 'H2-Alentejo', 'H2-Algarve',
       'H2-Centro', 'H2-Lisboa', 'H2-Norte'],
      dtype='object', name='name')
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io:Writing objective.
Writing continuous variables.

  Cost: 4.18 bn EUR/yr | Solar: 32.3 GW
Solving: solar cost = 50% ...


Index(['Alentejo', 'Algarve', 'Centro', 'Lisboa', 'Norte'], dtype='object', name='name')
Index(['hydro-Alentejo', 'hydro-Algarve', 'hydro-Centro', 'hydro-Lisboa',
       'hydro-Norte', 'onwind-Alentejo', 'solar-Alentejo', 'offwind-Alentejo',
       'onwind-Algarve', 'solar-Algarve', 'offwind-Algarve', 'onwind-Centro',
       'solar-Centro', 'offwind-Centro', 'onwind-Lisboa', 'solar-Lisboa',
       'offwind-Lisboa', 'onwind-Norte', 'solar-Norte', 'offwind-Norte'],
      dtype='object', name='name')
Index(['line-Norte-Centro', 'line-Centro-Lisboa', 'line-Centro-Alentejo',
       'line-Lisboa-Alentejo', 'line-Alentejo-Algarve'],
      dtype='object', name='name')
Index(['battery-Alentejo', 'battery-Algarve', 'battery-Centro',
       'battery-Lisboa', 'battery-Norte', 'H2-Alentejo', 'H2-Algarve',
       'H2-Centro', 'H2-Lisboa', 'H2-Norte'],
      dtype='object', name='name')
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io:Writing objective.
Writing continuous variables.

  Cost: 3.60 bn EUR/yr | Solar: 41.9 GW
Solving: solar cost = 25% ...


Index(['Alentejo', 'Algarve', 'Centro', 'Lisboa', 'Norte'], dtype='object', name='name')
Index(['hydro-Alentejo', 'hydro-Algarve', 'hydro-Centro', 'hydro-Lisboa',
       'hydro-Norte', 'onwind-Alentejo', 'solar-Alentejo', 'offwind-Alentejo',
       'onwind-Algarve', 'solar-Algarve', 'offwind-Algarve', 'onwind-Centro',
       'solar-Centro', 'offwind-Centro', 'onwind-Lisboa', 'solar-Lisboa',
       'offwind-Lisboa', 'onwind-Norte', 'solar-Norte', 'offwind-Norte'],
      dtype='object', name='name')
Index(['line-Norte-Centro', 'line-Centro-Lisboa', 'line-Centro-Alentejo',
       'line-Lisboa-Alentejo', 'line-Alentejo-Algarve'],
      dtype='object', name='name')
Index(['battery-Alentejo', 'battery-Algarve', 'battery-Centro',
       'battery-Lisboa', 'battery-Norte', 'H2-Alentejo', 'H2-Algarve',
       'H2-Centro', 'H2-Lisboa', 'H2-Norte'],
      dtype='object', name='name')
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io:Writing objective.
Writing continuous variables.

In [ ]:
df_sens.to_csv('sensitivity_results.csv')
print('Saved: sensitivity_results.csv')